# Experimento 3 — Adição da Variável Nitrogênio sem Balanceamento

Depois de analisar os resultados obtidos utilizando apenas as quatro variáveis iniciais, surgiu a necessidade de investigar se a adição de uma nova variável poderia ajudar o modelo a encontrar padrões ambientais mais complexos.

Até então, o modelo estava utilizando:

* temperatura;
* ortofosfato;
* país;
* tipo de corpo hídrico.

Os resultados anteriores mostraram que o modelo conseguia identificar relativamente bem a classe “Excelente”, mas ainda apresentava bastante dificuldade em reconhecer classes intermediárias e críticas, principalmente por conta do desbalanceamento do dataset e da complexidade ambiental envolvida na classificação da qualidade da água.

Diante disso, decidiu-se adicionar a variável nitrogênio ao conjunto de entrada do modelo.

É importante destacar que o nitrogênio NÃO foi utilizado diretamente na construção do `conama_status`, já que a forma como o nitrogênio aparece no dataset não possui equivalência perfeitamente validada com a forma como a Resolução CONAMA trabalha determinados compostos nitrogenados.

Mesmo assim, o nitrogênio ainda pode possuir relevância ambiental indireta.

Isso acontece porque parâmetros ambientais frequentemente possuem relações entre si. Em muitos contextos ambientais, alterações nos níveis de nitrogênio podem estar associadas a:

* matéria orgânica;
* presença de fertilizantes;
* eutrofização;
* degradação ambiental;
* alterações químicas da água.

Ou seja, mesmo não participando diretamente da regra de criação do rótulo, o nitrogênio ainda pode ajudar o modelo a encontrar padrões ambientais correlacionados com os parâmetros utilizados na construção do `conama_status`.

O objetivo deste experimento é justamente investigar se a adição dessa variável permite:

* melhorar a capacidade de generalização do modelo;
* reduzir parte das classificações incorretas;
* melhorar a identificação das classes intermediárias;
* reduzir a tendência do modelo de classificar excessivamente como “Excelente”.

Além disso, este experimento também ajuda a demonstrar uma característica muito importante do Machine Learning:

o modelo não precisa necessariamente utilizar apenas as variáveis que participaram diretamente da criação do rótulo.

O algoritmo pode aprender relações indiretas, padrões ocultos e associações matemáticas entre diferentes variáveis ambientais, desde que essas variáveis carreguem informação relevante para o problema analisado.

Neste experimento, o dataset continuará sem balanceamento, permitindo observar exclusivamente o impacto da adição da variável nitrogênio no comportamento do Random Forest.


## Preparação do ambiente


In [1]:
# IMPORTAÇÃO DE BIBLIOTECAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

SEED = 42

In [2]:
# DETECÇÃO DE AMBIENTE
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# CONFIGURAÇÃO DE CAMINHO E CARREGAMENTO
if IN_COLAB:
    print("Ambiente Google Colab detectado.")

    drive.mount('/content/drive')

    DATA_PATH = Path(
        "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/amostra_rotulada.parquet"
    )

else:
    print("Ambiente local/VS Code detectado.")

    DATA_PATH = Path("../../dataset/amostra_rotulada.parquet")

# LEITURA DO DATASET
df = pd.read_parquet(DATA_PATH)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente Google Colab detectado.
Mounted at /content/drive
Dataset Parquet carregado com sucesso.
Shape do dataset: (141399, 22)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status
0,Canada,CZPOH_1117,River,05-03-2012,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,...,93.116725,Good,1,1,1,1,3.7,1,5,Excelente
1,Canada,FISW_32,Lake,02-12-2003,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
2,Canada,CZPOH_1036,River,12-03-2018,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,...,100.000000,Excellent,1,1,1,1,3.7,1,5,Excelente
3,Canada,IEEA_10_32,Lake,08-06-2001,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
4,Canada,ES0910524,River,11-09-2012,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,...,92.882835,Good,1,1,1,1,2.0,1,5,Excelente


In [3]:
# preparando o ambiente para machine learning
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

In [4]:
# definição de x e y
X = df[
    [
        "Temperature (cel)",
        "Orthophosphate (mg/l)",
        "Country",
        "Waterbody Type",
        "Nitrogen (mg/l)"
    ]
]

y = df["conama_status"]

In [5]:
#train_test split

SEED = 42

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)


Treino: (113119, 5)
Teste: (28280, 5)


In [6]:
# pré-processamento

categorical_features = [
    "Country",
    "Waterbody Type"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [7]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        (
            "classifier",
            RandomForestClassifier(
                random_state=SEED
            )
        )
    ]
)

In [8]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country',
                                                   'Waterbody Type'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

## Avaliação das Métricas de Treino

Antes de analisar os resultados do conjunto de teste, também foi realizada a avaliação das métricas no conjunto de treino.

Essa etapa é importante porque permite comparar o comportamento do modelo entre treino e teste, ajudando na identificação de possíveis sinais de overfitting.

Por esse motivo, não basta apenas observar uma alta acurácia no treino. O ideal é que os resultados de treino e teste permaneçam relativamente próximos, indicando que o modelo conseguiu aprender padrões mais generalizáveis ao invés de apenas memorizar os dados.

Além da acurácia, também foram analisadas métricas como precision, recall e F1-score, permitindo observar como o modelo se comporta em cada classe individualmente.

In [9]:
y_train_pred = model.predict(X_train)

In [11]:
trains_accuracy = accuracy_score(y_train, y_train_pred)

print("Train Accuracy:")
print(trains_accuracy)

Train Accuracy:
0.9324605061925937


In [10]:
train_accuracy = accuracy_score(y_train, y_train_pred)

train_precision = precision_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_recall = recall_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_f1 = f1_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_confusion_matrix = confusion_matrix(
    y_train,
    y_train_pred
)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("Train Confusion Matrix:")
print(train_confusion_matrix)

Train Accuracy:
0.9324605061925937
Train Precision:
0.9325886108197478
Train Recall:
0.9324605061925937
Train F1:
0.9309900359703537
Train Confusion Matrix:
[[ 5422  1154     4   980]
 [   91 18160     3  3424]
 [    9   105   955    37]
 [   45  1786     2 80942]]


In [12]:
y_pred = model.predict(X_test)

In [13]:
# Teste com 5 variáveis - sem balanceamento
print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.76502828854314

Classification Report:
              precision    recall  f1-score   support

     Atenção       0.41      0.24      0.30      1890
         Boa       0.50      0.42      0.46      5419
     Crítica       0.16      0.06      0.09       277
   Excelente       0.84      0.91      0.87     20694

    accuracy                           0.77     28280
   macro avg       0.48      0.41      0.43     28280
weighted avg       0.74      0.77      0.75     28280


Confusion Matrix:
[[  454   679    24   733]
 [  315  2282    37  2785]
 [   54   116    17    90]
 [  277  1506    29 18882]]


# Resultados Obtidos — Experimento 3

Os resultados do terceiro experimento mostraram mudanças importantes no comportamento do modelo após a adição da variável nitrogênio.

Neste experimento, o dataset continuou sem balanceamento, permitindo analisar exclusivamente o impacto da nova variável no desempenho do Random Forest.

A acurácia de teste aumentou para aproximadamente 76%, representando uma melhora em relação aos experimentos anteriores realizados apenas com as quatro variáveis iniciais.

Além disso, observou-se uma melhora significativa na capacidade do modelo de identificar classes intermediárias, principalmente a classe `Boa`.

Nos experimentos anteriores, o modelo apresentava uma forte tendência de classificar grande parte das amostras como `Excelente`, consequência direta do desbalanceamento do dataset. Com a adição do nitrogênio, essa tendência diminuiu parcialmente, indicando que a nova variável ajudou o algoritmo a encontrar padrões ambientais mais complexos.

A quantidade de amostras da classe `Boa` classificadas incorretamente como `Excelente` reduziu consideravelmente, mostrando que o modelo passou a separar melhor diferentes níveis de qualidade da água.

A classe `Atenção` também apresentou melhora nas métricas de precisão e identificação, indicando um comportamento mais equilibrado do modelo.

Já a classe `Crítica` continuou apresentando dificuldades de reconhecimento. Isso acontece principalmente porque existem poucos exemplos dessa classe no dataset, fazendo com que o modelo tenha dificuldade em aprender padrões representativos suficientes.

Mesmo assim, houve uma pequena melhora na quantidade de acertos da classe crítica em comparação aos experimentos anteriores.

Outro ponto importante observado foi o aumento da acurácia no conjunto de treino, chegando a aproximadamente 93%, enquanto a acurácia no conjunto de teste permaneceu em torno de 76%.

Essa diferença relativamente alta entre treino e teste indica a presença de overfitting.

Isso significa que o Random Forest começou a aprender padrões muito específicos do conjunto de treino, reduzindo parte da sua capacidade de generalização para dados novos.

Ainda assim, o experimento demonstrou um resultado importante:

mesmo não participando diretamente da construção do `conama_status`, a variável nitrogênio conseguiu contribuir para a identificação indireta de padrões ambientais relacionados à qualidade da água.

Isso mostra que o Machine Learning pode aprender relações complexas entre variáveis ambientais, mesmo quando essas variáveis não fazem parte explicitamente da regra utilizada para criar o rótulo final.
